In [0]:
# =============================================================
# nb_silver_to_gold_manufacturers.py
# Layer     : Gold
# Audience  : Manufacturers / Engineering Team
# Purpose   : Orchestrates the Manufacturer device summary.
#
# Output table : gold/gold_manufacturer_device_summary
# Grain        : device_id (lifetime summary)
#
# Sources (Silver):
#   device_failures       → failure counts, severity, downtime
#   firmware_deployments  → success rate, rollbacks
#   device_snapshots      → battery health, signal drops
#   devices               → purchase_date, warranty_end_date
#   batches               → factory_name, factory_location
#   products              → product_name, device_category
#
# ADF params: storage_account
# =============================================================

# ── CELL 1: Widgets ───────────────────────────────────────────
dbutils.widgets.removeAll()
dbutils.widgets.text("storage_account", "petiot")


def abfss(container: str, path: str = "", storage_account: str | None = None) -> str:
    active_storage_account = storage_account or dbutils.widgets.get("storage_account")
    clean_path = path.lstrip("/")
    base = f"abfss://{container}@{active_storage_account}.dfs.core.windows.net"
    return f"{base}/{clean_path}" if clean_path else f"{base}/"

print("=" * 60)
print("  nb_silver_to_gold_manufacturers")
print("  Output: gold_manufacturer_device_summary")
print("  Grain : device_id (lifetime)")
print("=" * 60)

# ── CELL 2: Auth ──────────────────────────────────────────────
%run /Workspace/Shared/pet-analytics/shared/set_auth_context.py.ipynb

# ── CELL 3: Load specialists ──────────────────────────────────
%run /Workspace/Shared/pet-analytics/gold/gold_reader.ipynb
%run /Workspace/Shared/pet-analytics/gold/aggregator.ipynb
%run /Workspace/Shared/pet-analytics/gold/gold_writer.ipynb

# ── CELL 4: Read Silver fact tables ──────────────────────────
from pyspark.sql.functions import col, when, lit, current_date, datediff

print("\n[manufacturers] Reading Silver fact tables...")
failures_df  = read_device_failures()
fw_deploy_df = read_firmware_deployments()
snapshots_df = read_device_snapshots()

# Add deployment_id for counting (it was dropped in slim read)
# re-read with deployment_id for accurate count
fw_deploy_df = (
    spark.read.format("delta")
         .load(abfss("silver", "firmware_deployments"))
         .select("device_id", "deployment_id", "deployment_status",
                 "rollback_flag", "deployment_duration_seconds")
)

print(f"  device_failures       : {failures_df.count():,}")
print(f"  firmware_deployments  : {fw_deploy_df.count():,}")
print(f"  device_snapshots      : {snapshots_df.count():,}")

# ── CELL 5: Stage 1 — Aggregate each fact independently ───────
# Each fact → one row per device_id
print("\n[manufacturers] Stage 1: Aggregating facts per device...")

agg_fail = agg_failures_by_device(failures_df)
agg_fw   = agg_firmware_by_device(fw_deploy_df)
agg_snap = agg_device_health(snapshots_df)

print(f"  failure agg rows  : {agg_fail.count():,}")
print(f"  firmware agg rows : {agg_fw.count():,}")
print(f"  snapshot agg rows : {agg_snap.count():,}")

# ── CELL 6: Stage 2 — Full outer join aggregates ──────────────
# A device may have no failures but still have deployment history
# full_outer keeps all devices regardless of which facts they have
print("\n[manufacturers] Stage 2: Combining fact aggregates...")

fact_summary = (
    agg_fail
    .join(agg_fw,   "device_id", "full_outer")
    .join(agg_snap, "device_id", "left")
)
print(f"  Combined rows: {fact_summary.count():,}")

# ── CELL 7: Stage 3 — Enrich with dimension labels ────────────
print("\n[manufacturers] Stage 3: Enriching with dimensions...")

devices_dim  = read_devices()
batches_dim  = read_batches()
products_dim = read_products()

gold_df = (
    fact_summary
    .join(devices_dim,  "device_id",  "left")
    .join(batches_dim,  "batch_id",   "left")
    .join(products_dim, "product_id", "left")
)

# ── CELL 8: Derive warranty status ────────────────────────────
gold_df = (
    gold_df
    .withColumn("warranty_status",
        when(col("warranty_end_date") >= current_date(), lit("Active"))
        .otherwise(lit("Expired")))
    .withColumn("days_since_purchase",
        datediff(current_date(), col("purchase_date")))
)

# ── CELL 9: Apply null defaults ───────────────────────────────
gold_df = apply_null_defaults(gold_df, "gold_manufacturer_device_summary")

# ── CELL 10: Add audit + write ────────────────────────────────
gold_df     = add_gold_audit(gold_df)
final_count = write_gold(gold_df, "gold_manufacturer_device_summary")

# ── CELL 11: Preview ──────────────────────────────────────────
print("\n[manufacturers] Sample output:")
spark.read.format("delta") \
    .load(abfss("gold","gold_manufacturer_device_summary")) \
    .select("device_id","product_name","factory_name","factory_location",
            "total_failures","fw_success_rate_pct",
            "avg_battery_pct","warranty_status","is_crash_prone") \
    .show(5, truncate=False)

print(f"\n{'='*60}")
print(f"  Gold complete: gold_manufacturer_device_summary")
print(f"  Rows: {final_count:,}")
print(f"{'='*60}")

dbutils.notebook.exit(f"SUCCESS|gold_manufacturer_device_summary|{final_count}")